In [62]:
import pandas as pd

# Load DataSpeciesPerCode
data_path = r"C:\Users\jdelhoyo\PhD\Study cases\Genissiat\RV Characterization\repo-github\data\DefinitiveRV4(withconifers).csv"
df_species = pd.read_csv(data_path)

print(f"Loaded DataSpeciesPerCode: {df_species.shape[0]} rows, {df_species.shape[1]} columns")
print(df_species.head())

Loaded DataSpeciesPerCode: 88 rows, 153 columns
   Id_Data  Id_Place Basin Sub_Basin Stand_Code    Bank     Cod_Plg  \
0        1         1  Arve      Arve       A-A1   Left    A-A1-Left   
1        2         1  Arve      Arve       A-A1  Right   A-A1-Right   
2        3         2  Arve      Arve       A-A2   Left    A-A2-Left   
3        4         2  Arve      Arve       A-A2  Right   A-A2-Right   
4        5         3  Arve      Arve       A-A3   Left    A-A3-Left   

   Standing_Dead_Trees  Dead_Wood  Regeneration  ...  sp_11_10-20  \
0                    1          1             1  ...          0.0   
1                    3          3             3  ...          0.0   
2                    1          1             1  ...          0.0   
3                    1          1             1  ...          NaN   
4                    1          2             1  ...          0.0   

   sp_11_20-30  sp_11_30-40  sp_11_40-50  sp_11_>50  sp_12_10-20  sp_12_20-30  \
0          0.0          0.0  

In [63]:
from pathlib import Path

# Define output directory for results
output_dir = Path(r"C:\Users\jdelhoyo\PhD\Study cases\Genissiat\RV Characterization\repo-github\data\Results\VegetationIndex")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}")
print(f"Directory exists: {output_dir.exists()}")

Output directory: C:\Users\jdelhoyo\PhD\Study cases\Genissiat\RV Characterization\repo-github\data\Results\VegetationIndex
Directory exists: True


# 1. Data Structure and Exploration

Define species to use and explore the data structure.

In [64]:
# Define valid species codes
valid_species = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19, 22, 25]

species_names = {
    1: 'Abies',
    2: 'Populus',
    3: 'Salix',
    4: 'Alnus',
    5: 'Fraxinus',
    6: 'Fagus',
    7: 'Sorbus',
    8: 'Quercus',
    9: 'Ulmus',
    10: 'Larix',
    11: 'Pinus',
    12: 'Picea',
    13: 'Prunus',
    15: 'Acer',
    16: 'Castanea',
    18: 'Tilia',
    19: 'Robinia',
    22: 'Aesculus',
    25: 'Juglans'
}

print(f"Valid species: {valid_species}")
print(f"Total valid species: {len(valid_species)}")
print(f"\nData shape: {df_species.shape}")
print(f"Columns in data: {df_species.columns.tolist()[:10]}...")  # Show first 10 columns

Valid species: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19, 22, 25]
Total valid species: 19

Data shape: (88, 153)
Columns in data: ['Id_Data', 'Id_Place', 'Basin', 'Sub_Basin', 'Stand_Code', 'Bank', 'Cod_Plg', 'Standing_Dead_Trees', 'Dead_Wood', 'Regeneration']...


In [65]:
# Identify species presence columns (sp_1, sp_2, ..., sp_29)
species_presence_cols = [col for col in df_species.columns if col.startswith('sp_') and col.split('_')[1].isdigit()]
species_presence_cols.sort(key=lambda x: int(x.split('_')[1]))

# Identify diameter class columns (sp_X_10-20, sp_X_20-30, etc.)
diameter_classes = ['10-20', '20-30', '30-40', '40-50', '>50']
species_diameter_cols = {}

for sp_code in valid_species:
    sp_diameter_cols = []
    for diameter_class in diameter_classes:
        col_name = f'sp_{sp_code}_{diameter_class}'
        if col_name in df_species.columns:
            sp_diameter_cols.append(col_name)
    if sp_diameter_cols:
        species_diameter_cols[sp_code] = sp_diameter_cols

print(f"Species presence columns found: {len(species_presence_cols)}")
print(f"Example: {species_presence_cols[:5]}")
print(f"\nSpecies with diameter class columns: {len(species_diameter_cols)}")
print(f"Example: sp_1 -> {species_diameter_cols.get(1, 'None')}")

# Filter to only valid species presence columns
valid_sp_presence_cols = [f'sp_{sp}' for sp in valid_species if f'sp_{sp}' in species_presence_cols]
print(f"\nValid species presence columns: {len(valid_sp_presence_cols)}")
print(f"Columns: {valid_sp_presence_cols}")

Species presence columns found: 93
Example: ['sp_1', 'sp_1_10-20', 'sp_1_20-30', 'sp_1_30-40', 'sp_1_40-50']

Species with diameter class columns: 19
Example: sp_1 -> ['sp_1_10-20', 'sp_1_20-30', 'sp_1_30-40', 'sp_1_40-50', 'sp_1_>50']

Valid species presence columns: 19
Columns: ['sp_1', 'sp_2', 'sp_3', 'sp_4', 'sp_5', 'sp_6', 'sp_7', 'sp_8', 'sp_9', 'sp_10', 'sp_11', 'sp_12', 'sp_13', 'sp_15', 'sp_16', 'sp_18', 'sp_19', 'sp_22', 'sp_25']


In [66]:
# Prepare analysis dataframe with only valid species
# Keep Cod_Plg column and valid species presence/diameter columns

cols_to_keep = ['Cod_Plg'] if 'Cod_Plg' in df_species.columns else []
cols_to_keep.extend(valid_sp_presence_cols)

# Add diameter class columns
for sp_code in valid_species:
    for diameter_class in diameter_classes:
        col_name = f'sp_{sp_code}_{diameter_class}'
        if col_name in df_species.columns:
            cols_to_keep.append(col_name)

# Create clean dataframe
df_analysis = df_species[cols_to_keep].copy()

print(f"Analysis dataframe shape: {df_analysis.shape}")
print(f"Total columns after filtering: {len(df_analysis.columns)}")
print(f"\nFirst few rows:")
print(df_analysis.head())
print(f"\nColumn summary:")
print(f"  - ID column: Cod_Plg")
print(f"  - Species presence columns: {len(valid_sp_presence_cols)}")
print(f"  - Diameter class columns: {len(cols_to_keep) - len(valid_sp_presence_cols) - 1}")
print(f"  - Total: {len(cols_to_keep)-1} species data columns")

Analysis dataframe shape: (88, 84)
Total columns after filtering: 84

First few rows:
      Cod_Plg  sp_1  sp_2  sp_3  sp_4  sp_5  sp_6  sp_7  sp_8  sp_9  ...  \
0   A-A1-Left     1     0     1     1     1     0     0     0     0  ...   
1  A-A1-Right     0     0     1     1     0     0     1     0     0  ...   
2   A-A2-Left     0     1     1     1     0     0     0     0     0  ...   
3  A-A2-Right     0     1     1     1     0     0     0     0     1  ...   
4   A-A3-Left     0     1     0     0     1     0     0     0     0  ...   

   sp_16_20-30  sp_16_30-40  sp_18_10-20  sp_19_10-20  sp_19_20-30  \
0            0            0            0            0            0   
1            0            0            0            0            0   
2            0            0            0            1            0   
3            0            0            0            1            1   
4            0            0            0            0            0   

   sp_22_10-20  sp_22_30-40  sp_25_1

# 2. Calculate Species Richness Index

Calculate richness metrics from species presence data.

In [67]:
# Step 1: Calculate species richness (S)
# Count how many valid species are present (value > 0) in each row

df_analysis['S'] = (df_analysis[valid_sp_presence_cols] > 0).sum(axis=1)

print("Step 1: Species Richness (S)")
print("="*60)
print(f"S = count of valid species present (sp_X > 0)")
print(f"\nRichness statistics:")
print(f"  - Min: {df_analysis['S'].min()}")
print(f"  - Max: {df_analysis['S'].max()}")
print(f"  - Mean: {df_analysis['S'].mean():.2f}")
print(f"  - Median: {df_analysis['S'].median():.0f}")
print(f"\nRichness distribution:")
print(df_analysis['S'].value_counts().sort_index())
print(f"\nFirst 10 rows:")
print(df_analysis[['Cod_Plg', 'S']].head(10))

Step 1: Species Richness (S)
S = count of valid species present (sp_X > 0)

Richness statistics:
  - Min: 2
  - Max: 9
  - Mean: 5.25
  - Median: 6

Richness distribution:
S
2     3
3    12
4    15
5    13
6    26
7    14
8     4
9     1
Name: count, dtype: int64

First 10 rows:
      Cod_Plg  S
0   A-A1-Left  7
1  A-A1-Right  5
2   A-A2-Left  4
3  A-A2-Right  6
4   A-A3-Left  3
5  A-A3-Right  3
6   A-A4-Left  4
7  A-A4-Right  3
8   A-A5-Left  8
9  A-A5-Right  7


In [68]:
# Step 2: Calculate maximum richness observed
S_max = df_analysis['S'].max()

print("\nStep 2: Maximum Richness Observed")
print("="*60)
print(f"S_max = {S_max}")
print(f"\nThis is the maximum number of species present in any single plot")


Step 2: Maximum Richness Observed
S_max = 9

This is the maximum number of species present in any single plot


In [69]:
# Step 3: Calculate logarithmic proxy of richness
# H_sp_proxy = ln(S) if S > 0, else 0

import numpy as np

df_analysis['H_sp_proxy'] = df_analysis['S'].apply(lambda x: np.log(x) if x > 0 else 0)

print("\nStep 3: Logarithmic Proxy of Richness (H_sp_proxy)")
print("="*60)
print(f"H_sp_proxy = ln(S) if S > 0, else 0")
print(f"\nProxy statistics:")
print(f"  - Min: {df_analysis['H_sp_proxy'].min():.4f}")
print(f"  - Max: {df_analysis['H_sp_proxy'].max():.4f}")
print(f"  - Mean: {df_analysis['H_sp_proxy'].mean():.4f}")
print(f"  - Median: {df_analysis['H_sp_proxy'].median():.4f}")
print(f"\nFirst 10 rows:")
print(df_analysis[['Cod_Plg', 'S', 'H_sp_proxy']].head(10))


Step 3: Logarithmic Proxy of Richness (H_sp_proxy)
H_sp_proxy = ln(S) if S > 0, else 0

Proxy statistics:
  - Min: 0.6931
  - Max: 2.1972
  - Mean: 1.6059
  - Median: 1.7918

First 10 rows:
      Cod_Plg  S  H_sp_proxy
0   A-A1-Left  7    1.945910
1  A-A1-Right  5    1.609438
2   A-A2-Left  4    1.386294
3  A-A2-Right  6    1.791759
4   A-A3-Left  3    1.098612
5  A-A3-Right  3    1.098612
6   A-A4-Left  4    1.386294
7  A-A4-Right  3    1.098612
8   A-A5-Left  8    2.079442
9  A-A5-Right  7    1.945910


In [70]:
# Step 4: Calculate normalized richness proxy (J_sp_proxy)
# Normalize between 0 and 1

def calculate_j_sp_proxy(row, s_max):
    s = row['S']
    h_sp = row['H_sp_proxy']
    
    if s_max <= 1:
        # If max richness is 0 or 1, simple binary
        return 1 if s > 0 else 0
    else:
        # Normal case: normalize by ln(S_max)
        return h_sp / np.log(s_max)

df_analysis['J_sp_proxy'] = df_analysis.apply(lambda row: calculate_j_sp_proxy(row, S_max), axis=1)

print("\nStep 4: Normalized Richness Proxy (J_sp_proxy)")
print("="*60)
print(f"Normalization formula:")
if S_max <= 1:
    print(f"  - S_max = {S_max} (≤ 1): J_sp_proxy = 1 if S > 0, else 0")
else:
    print(f"  - S_max = {S_max} (> 1): J_sp_proxy = H_sp_proxy / ln({S_max})")
    print(f"  - Normalization factor: ln({S_max}) = {np.log(S_max):.4f}")

print(f"\nNormalized proxy statistics:")
print(f"  - Min: {df_analysis['J_sp_proxy'].min():.4f}")
print(f"  - Max: {df_analysis['J_sp_proxy'].max():.4f}")
print(f"  - Mean: {df_analysis['J_sp_proxy'].mean():.4f}")
print(f"  - Median: {df_analysis['J_sp_proxy'].median():.4f}")

print(f"\nFull richness summary (first 15 rows):")
summary_cols = ['Cod_Plg', 'S', 'H_sp_proxy', 'J_sp_proxy']
print(df_analysis[summary_cols].head(15).to_string(index=False))

print(f"\n✓ All richness indices calculated and normalized between 0 and 1")


Step 4: Normalized Richness Proxy (J_sp_proxy)
Normalization formula:
  - S_max = 9 (> 1): J_sp_proxy = H_sp_proxy / ln(9)
  - Normalization factor: ln(9) = 2.1972

Normalized proxy statistics:
  - Min: 0.3155
  - Max: 1.0000
  - Mean: 0.7309
  - Median: 0.8155

Full richness summary (first 15 rows):
   Cod_Plg  S  H_sp_proxy  J_sp_proxy
 A-A1-Left  7    1.945910    0.885622
A-A1-Right  5    1.609438    0.732487
 A-A2-Left  4    1.386294    0.630930
A-A2-Right  6    1.791759    0.815465
 A-A3-Left  3    1.098612    0.500000
A-A3-Right  3    1.098612    0.500000
 A-A4-Left  4    1.386294    0.630930
A-A4-Right  3    1.098612    0.500000
 A-A5-Left  8    2.079442    0.946395
A-A5-Right  7    1.945910    0.885622
 A-A6-Left  7    1.945910    0.885622
A-A6-Right  5    1.609438    0.732487
 A-A7-Left  6    1.791759    0.815465
A-A7-Right  9    2.197225    1.000000
 A-A8-Left  8    2.079442    0.946395

✓ All richness indices calculated and normalized between 0 and 1


In [71]:
# Export richness results
richness_export = df_analysis[['Cod_Plg', 'S', 'H_sp_proxy', 'J_sp_proxy']].copy()
richness_file = output_dir / 'RichnessIndex_Normalized.csv'
richness_export.to_csv(richness_file, index=False)
print(f"\n✓ Exported richness data to: {richness_file.name}")


✓ Exported richness data to: RichnessIndex_Normalized.csv


# 3. Calculate Diameter Diversity Index

Calculate Shannon diversity and equitability across diameter classes.

In [72]:
# Step 1: Calculate number of species per diameter class
# For each row and each diameter class, count species with presence > 0

diameter_classes_list = ['10-20', '20-30', '30-40', '40-50', '>50']

for diam_class in diameter_classes_list:
    # Get all species_X_diam_class columns for valid species
    cols_for_class = []
    for sp_code in valid_species:
        col_name = f'sp_{sp_code}_{diam_class}'
        if col_name in df_analysis.columns:
            cols_for_class.append(col_name)
    
    # Count species present in this class (value > 0)
    col_name_n = f'n_{diam_class}'
    df_analysis[col_name_n] = (df_analysis[cols_for_class] > 0).sum(axis=1)

print("Step 1: Species count per diameter class")
print("="*60)
print("For each row, counted species present in each diameter class:")
print(f"\nColumns added: n_10-20, n_20-30, n_30-40, n_40-50, n_>50")
print(f"\nExample (first 10 rows):")
count_cols = ['Cod_Plg'] + [f'n_{dc}' for dc in diameter_classes_list]
print(df_analysis[count_cols].head(10).to_string(index=False))

Step 1: Species count per diameter class
For each row, counted species present in each diameter class:

Columns added: n_10-20, n_20-30, n_30-40, n_40-50, n_>50

Example (first 10 rows):
   Cod_Plg  n_10-20  n_20-30  n_30-40  n_40-50  n_>50
 A-A1-Left        4        4        1        1      0
A-A1-Right        2        2        2        1      1
 A-A2-Left        4        2        1        0      0
A-A2-Right        4        4        0        1      0
 A-A3-Left        3        1        0        1      0
A-A3-Right        2        2        1        1      0
 A-A4-Left        4        2        1        0      0
A-A4-Right        3        2        0        1      0
 A-A5-Left        5        6        1        1      0
A-A5-Right        4        5        2        0      0


In [73]:
# Step 2: Calculate Shannon diameter and equitability (J_dbh)

def calculate_shannon_diameter(row):
    """
    Calculate Shannon diversity (H_dbh) and equitability (J_dbh) 
    across diameter classes
    """
    # Get active class counts
    counts = [
        row['n_10-20'],
        row['n_20-30'],
        row['n_30-40'],
        row['n_40-50'],
        row['n_>50']
    ]
    
    # Get active (non-zero) classes
    active = [c for c in counts if c > 0]
    C = len(active)  # Number of active classes
    
    if C == 0:
        # No diameter classes present
        return 0, 0
    elif C == 1:
        # Only one diameter class: no diversity
        return 0, np.nan
    else:
        # C >= 2: calculate Shannon
        total = sum(active)
        proportions = [a / total for a in active]
        
        # Shannon: H = -sum(p_i * ln(p_i))
        h_dbh = -sum(p * np.log(p) for p in proportions if p > 0)
        
        # Equitability: J = H / ln(C)
        j_dbh = h_dbh / np.log(C)
        
        return h_dbh, j_dbh

# Apply calculation to each row
results = df_analysis.apply(calculate_shannon_diameter, axis=1, result_type='expand')
df_analysis['H_dbh'] = results[0]
df_analysis['J_dbh'] = results[1]

print("\nStep 2: Shannon Diameter Diversity (H_dbh) and Equitability (J_dbh)")
print("="*60)
print("Formula:")
print("  - If C = 0: H_dbh = 0, J_dbh = 0")
print("  - If C = 1: H_dbh = 0, J_dbh = NaN")
print("  - If C ≥ 2: H_dbh = -Σ(p_i * ln(p_i)), J_dbh = H_dbh / ln(C)")
print(f"\nStatistics:")
print(f"  H_dbh - Min: {df_analysis['H_dbh'].min():.4f}, Max: {df_analysis['H_dbh'].max():.4f}, Mean: {df_analysis['H_dbh'].mean():.4f}")
print(f"  J_dbh - Min: {df_analysis['J_dbh'].min():.4f}, Max: {df_analysis['J_dbh'].max():.4f}, Mean: {df_analysis['J_dbh'].mean():.4f}")

print(f"\nExample (first 15 rows):")
example_cols = ['Cod_Plg'] + [f'n_{dc}' for dc in diameter_classes_list] + ['H_dbh', 'J_dbh']
print(df_analysis[example_cols].head(15).to_string(index=False))

print(f"\n✓ Shannon diameter indices calculated")


Step 2: Shannon Diameter Diversity (H_dbh) and Equitability (J_dbh)
Formula:
  - If C = 0: H_dbh = 0, J_dbh = 0
  - If C = 1: H_dbh = 0, J_dbh = NaN
  - If C ≥ 2: H_dbh = -Σ(p_i * ln(p_i)), J_dbh = H_dbh / ln(C)

Statistics:
  H_dbh - Min: 0.5004, Max: 1.5596, Mean: 1.0573
  J_dbh - Min: 0.7219, Max: 1.0000, Mean: 0.8994

Example (first 15 rows):
   Cod_Plg  n_10-20  n_20-30  n_30-40  n_40-50  n_>50    H_dbh    J_dbh
 A-A1-Left        4        4        1        1      0 1.193550 0.860964
A-A1-Right        2        2        2        1      1 1.559581 0.969022
 A-A2-Left        4        2        1        0      0 0.955700 0.869916
A-A2-Right        4        4        0        1      0 0.964963 0.878347
 A-A3-Left        3        1        0        1      0 0.950271 0.864974
A-A3-Right        2        2        1        1      0 1.329661 0.959148
 A-A4-Left        4        2        1        0      0 0.955700 0.869916
A-A4-Right        3        2        0        1      0 1.011404 0.920620
 A

In [74]:
# Export diameter diversity results
diameter_export_cols = ['Cod_Plg'] + [f'n_{dc}' for dc in diameter_classes_list] + ['H_dbh', 'J_dbh']
diameter_export = df_analysis[diameter_export_cols].copy()
diameter_file = output_dir / 'DiameterDiversity_Index.csv'
diameter_export.to_csv(diameter_file, index=False)
print(f"\n✓ Exported diameter diversity data to: {diameter_file.name}")


✓ Exported diameter diversity data to: DiameterDiversity_Index.csv


In [75]:
# Step 3: Fix J_dbh for single diameter class cases (C=1)
# Calculate J_min from rows with C >= 2

# First, identify rows with C >= 1 and C >= 2
def get_active_classes_count(row):
    counts = [row['n_10-20'], row['n_20-30'], row['n_30-40'], row['n_40-50'], row['n_>50']]
    active = [c for c in counts if c > 0]
    return len(active)

df_analysis['C'] = df_analysis.apply(get_active_classes_count, axis=1)

# Find J_min from rows with C >= 2
j_dbh_c_ge_2 = df_analysis[df_analysis['C'] >= 2]['J_dbh']
if len(j_dbh_c_ge_2) > 0:
    J_min = j_dbh_c_ge_2.min()
else:
    J_min = 0

print("Step 3: Fix J_dbh for single diameter class cases")
print("="*60)
print(f"Rows with C=0: {len(df_analysis[df_analysis['C']==0])}")
print(f"Rows with C=1: {len(df_analysis[df_analysis['C']==1])}")
print(f"Rows with C>=2: {len(df_analysis[df_analysis['C']>=2])}")
print(f"J_min (from C>=2): {J_min:.6f}")

# Replace NaN values in J_dbh (from C=1 cases) with J_min
df_analysis.loc[df_analysis['C'] == 1, 'J_dbh'] = J_min

print(f"\nAfter replacement:")
print(f"  J_dbh - Min: {df_analysis['J_dbh'].min():.6f}, Max: {df_analysis['J_dbh'].max():.6f}")
print(f"  NaN values in J_dbh: {df_analysis['J_dbh'].isna().sum()}")
print(f"\n✓ Fixed J_dbh values for C=1 cases with J_min")

Step 3: Fix J_dbh for single diameter class cases
Rows with C=0: 0
Rows with C=1: 0
Rows with C>=2: 88
J_min (from C>=2): 0.721928

After replacement:
  J_dbh - Min: 0.721928, Max: 1.000000
  NaN values in J_dbh: 0

✓ Fixed J_dbh values for C=1 cases with J_min


In [76]:
# Step 4: Calculate combined structural index
# D_comb = 0.5 * J_sp_proxy + 0.5 * J_dbh

alpha = 0.5
df_analysis['D_comb'] = alpha * df_analysis['J_sp_proxy'] + (1 - alpha) * df_analysis['J_dbh']

print("\nStep 4: Combined Structural Index (D_comb)")
print("="*60)
print(f"Formula: D_comb = {alpha} * J_sp_proxy + {1-alpha} * J_dbh")
print(f"\nBalances:")
print(f"  - Species richness relative: {alpha*100:.0f}%")
print(f"  - Diameter diversity/equitability: {(1-alpha)*100:.0f}%")

print(f"\nStatistics:")
print(f"  Min: {df_analysis['D_comb'].min():.4f}")
print(f"  Max: {df_analysis['D_comb'].max():.4f}")
print(f"  Mean: {df_analysis['D_comb'].mean():.4f}")
print(f"  Median: {df_analysis['D_comb'].median():.4f}")

print(f"\nExample (first 10 rows):")
example_cols = ['Cod_Plg', 'J_sp_proxy', 'J_dbh', 'D_comb']
print(df_analysis[example_cols].head(10).to_string(index=False))


Step 4: Combined Structural Index (D_comb)
Formula: D_comb = 0.5 * J_sp_proxy + 0.5 * J_dbh

Balances:
  - Species richness relative: 50%
  - Diameter diversity/equitability: 50%

Statistics:
  Min: 0.6169
  Max: 0.9428
  Mean: 0.8152
  Median: 0.8238

Example (first 10 rows):
   Cod_Plg  J_sp_proxy    J_dbh   D_comb
 A-A1-Left    0.885622 0.860964 0.873293
A-A1-Right    0.732487 0.969022 0.850755
 A-A2-Left    0.630930 0.869916 0.750423
A-A2-Right    0.815465 0.878347 0.846906
 A-A3-Left    0.500000 0.864974 0.682487
A-A3-Right    0.500000 0.959148 0.729574
 A-A4-Left    0.630930 0.869916 0.750423
A-A4-Right    0.500000 0.920620 0.710310
 A-A5-Left    0.946395 0.807165 0.876780
A-A5-Right    0.885622 0.943189 0.914405


In [77]:
# Step 5: Calculate percentiles and classify
# Create qualitative classes

p25 = df_analysis['D_comb'].quantile(0.25)
p50 = df_analysis['D_comb'].quantile(0.50)
p75 = df_analysis['D_comb'].quantile(0.75)

def classify_diversity(value):
    if value < p25:
        return 'Low'
    elif value < p50:
        return 'Medium'
    elif value < p75:
        return 'High'
    else:
        return 'Very high'

df_analysis['Clase_div_global'] = df_analysis['D_comb'].apply(classify_diversity)

print("\nStep 5: Qualitative Classification")
print("="*60)
print(f"Percentiles:")
print(f"  - p25 (25%): {p25:.4f}")
print(f"  - p50 (50%): {p50:.4f}")
print(f"  - p75 (75%): {p75:.4f}")

print(f"\nClasses:")
print(f"  'Low':       D_comb < {p25:.4f}")
print(f"  'Medium':    {p25:.4f} <= D_comb < {p50:.4f}")
print(f"  'High':      {p50:.4f} <= D_comb < {p75:.4f}")
print(f"  'Very high': D_comb >= {p75:.4f}")

print(f"\nClass distribution:")
print(df_analysis['Clase_div_global'].value_counts().sort_index())

print(f"\nExample (first 15 rows):")
example_cols = ['Cod_Plg', 'D_comb', 'Clase_div_global']
print(df_analysis[example_cols].head(15).to_string(index=False))

print(f"\n✓ Classification completed")


Step 5: Qualitative Classification
Percentiles:
  - p25 (25%): 0.7725
  - p50 (50%): 0.8238
  - p75 (75%): 0.8693

Classes:
  'Low':       D_comb < 0.7725
  'Medium':    0.7725 <= D_comb < 0.8238
  'High':      0.8238 <= D_comb < 0.8693
  'Very high': D_comb >= 0.8693

Class distribution:
Clase_div_global
High         21
Low          21
Medium       23
Very high    23
Name: count, dtype: int64

Example (first 15 rows):
   Cod_Plg   D_comb Clase_div_global
 A-A1-Left 0.873293        Very high
A-A1-Right 0.850755             High
 A-A2-Left 0.750423              Low
A-A2-Right 0.846906             High
 A-A3-Left 0.682487              Low
A-A3-Right 0.729574              Low
 A-A4-Left 0.750423              Low
A-A4-Right 0.710310              Low
 A-A5-Left 0.876780        Very high
A-A5-Right 0.914405        Very high
 A-A6-Left 0.920028        Very high
A-A6-Right 0.753442              Low
 A-A7-Left 0.834129             High
A-A7-Right 0.875000        Very high
 A-A8-Left 0.877495  

In [78]:
# Final export: Complete results with all required columns

final_output_cols = [
    'Cod_Plg',
    'S',
    'H_sp_proxy',
    'J_sp_proxy',
    'H_dbh',
    'J_dbh',
    'D_comb',
    'Clase_div_global'
]

df_final = df_analysis[final_output_cols].copy()

# Export to CSV
final_file = output_dir / 'VegetationIndex_Complete.csv'
df_final.to_csv(final_file, index=False)

print("\n" + "="*60)
print("FINAL RESULTS EXPORTED")
print("="*60)
print(f"\nFile: {final_file.name}")
print(f"Rows: {len(df_final):,}")
print(f"Columns: {len(df_final.columns)}")

print(f"\nColumn structure:")
for i, col in enumerate(final_output_cols, 1):
    print(f"  {i}. {col}")

print(f"\nSummary statistics:")
print(df_final.describe().round(4))

print(f"\nClass distribution (Clase_div_global):")
print(df_final['Clase_div_global'].value_counts().sort_index())

print(f"\n✓ Complete vegetation index calculated and exported!")
print(f"✓ File saved: {final_file}")


FINAL RESULTS EXPORTED

File: VegetationIndex_Complete.csv
Rows: 88
Columns: 8

Column structure:
  1. Cod_Plg
  2. S
  3. H_sp_proxy
  4. J_sp_proxy
  5. H_dbh
  6. J_dbh
  7. D_comb
  8. Clase_div_global

Summary statistics:
             S  H_sp_proxy  J_sp_proxy    H_dbh    J_dbh   D_comb
count  88.0000     88.0000     88.0000  88.0000  88.0000  88.0000
mean    5.2500      1.6059      0.7309   1.0573   0.8994   0.8152
std     1.5848      0.3403      0.1549   0.2256   0.0616   0.0709
min     2.0000      0.6931      0.3155   0.5004   0.7219   0.6169
25%     4.0000      1.3863      0.6309   0.9543   0.8626   0.7725
50%     6.0000      1.7918      0.8155   1.0640   0.9141   0.8238
75%     6.0000      1.7918      0.8155   1.2203   0.9467   0.8693
max     9.0000      2.1972      1.0000   1.5596   1.0000   0.9428

Class distribution (Clase_div_global):
Clase_div_global
High         21
Low          21
Medium       23
Very high    23
Name: count, dtype: int64

✓ Complete vegetation index ca

In [79]:
# Class count summary

print("\n" + "="*60)
print("CLASS DISTRIBUTION SUMMARY")
print("="*60)

# Debug: check unique values in the class column
print(f"\nUnique values in Clase_div_global column:")
print(df_final['Clase_div_global'].unique())
print(f"\nValue counts (unsorted):")
print(df_final['Clase_div_global'].value_counts())

class_counts = df_final['Clase_div_global'].value_counts()
total_records = len(df_final)

print(f"\nTotal records: {total_records:,}\n")

# Sort in order: Low, Medium, High, Very high
class_order = ['Low', 'Medium', 'High', 'Very high']
for class_name in class_order:
    if class_name in class_counts.index:
        count = class_counts[class_name]
        percentage = (count / total_records) * 100
        print(f"  {class_name:12} : {count:6,} plots  ({percentage:6.2f}%)")
    else:
        print(f"  {class_name:12} :      0 plots  (  0.00%)")

print(f"\n✓ Count completed")


CLASS DISTRIBUTION SUMMARY

Unique values in Clase_div_global column:
['Very high' 'High' 'Low' 'Medium']

Value counts (unsorted):
Clase_div_global
Very high    23
Medium       23
High         21
Low          21
Name: count, dtype: int64

Total records: 88

  Low          :     21 plots  ( 23.86%)
  Medium       :     23 plots  ( 26.14%)
  High         :     21 plots  ( 23.86%)
  Very high    :     23 plots  ( 26.14%)

✓ Count completed
